# Clipper Tahap 2 Backend Penuh - Kaggle Notebook
Jalankan notebook ini di Kaggle dengan GPU T4 x2. Ini memuat keseluruhan backend FastAPI, FFmpeg, yt-dlp, Gemini, dan Qwen2.5-VL-3B secara native.

In [ ]:
# Copy folder clipper dari Dataset ke direktori kerja Kaggle (/kaggle/working/)
# PASTIKAN Anda menyesuaikan 'clipper-backend' (atau nama dataset Anda)
!cp -a /kaggle/input/datasets/xxxjho/backend-clipper/. /kaggle/working/
!pip install -r requirements.txt
!pip install pyngrok transformers accelerate qwen-vl-utils pillow
!mkdir -p /kaggle/working/output
# Pastikan GPU aktif; bila False, VLM akan fallback ke mock dan frame tidak divalidasi
!python -c "import torch; print('CUDA tersedia:', torch.cuda.is_available())"

In [ ]:
import os
os.environ["OUTPUT_DIR"] = "/kaggle/working/output"
os.environ["VLM_PROVIDER"] = "transformers"
os.environ["GEMINI_API_KEY"] = "your_api_key_here"

# Fail-fast config
os.environ["YTDLP_TIMEOUT_SEC"] = "45"
os.environ["YTDLP_RETRIES"] = "1"
os.environ["YTDLP_MAX_RETRIES"] = "1"

# Proxy SOCKS5/HTTP:
# os.environ["YTDLP_PROXY"] = "socks5://127.0.0.1:1080"

In [ ]:
from pyngrok import ngrok
# Masukkan Authtoken Ngrok Anda (wajib untuk mengekspos web UI ke HP Android Anda)
ngrok.set_auth_token("YOUR_NGROK_AUTH_TOKEN")
public_url = ngrok.connect(5000)
print("=================================================")
print(f"BUKA LINK INI DI BROWSER HP ANDA: {public_url}")
print("=================================================")

In [ ]:
!uvicorn app.main:app --host 0.0.0.0 --port 5000

### Opsi B: Arsitektur Hybrid (Sangat Direkomendasikan Jika YouTube di Kaggle Terblokir)
Jika Kaggle terus mengalami `yt-dlp timeout`, sebaiknya gunakan Kaggle **HANYA** sebagai backend AI (Oracle) dan jalankan proses utama (Frontend + Unduh Video) di HP Termux/PC Anda.

Caranya:
1. Di cell atas, JANGAN jalankan `uvicorn app.main:app`. Melainkan jalankan: `!python kaggle/kaggle_oracle.py`
2. Di Termux/PC Anda, atur `.env`:
   ```
   VLM_PROVIDER=oracle
   VLM_ORACLE_URL=https://(url-ngrok-dari-kaggle).ngrok-free.app
   ```
   (Atau cukup set `NGROK_URL` di `.env` — kini otomatis terbaca sebagai `VLM_ORACLE_URL`.)
3. Jalankan `uvicorn app.main:app --port 5000` di Termux/PC Anda.

Catatan: URL stream googlevideo terikat IP mesin yang me-resolve-nya (403 Forbidden dari IP lain). Karena itu proses unduh video TIDAK bisa dijalankan dari Kaggle; unduhan harus tetap berjalan di Termux/PC Anda.

In [ ]:
# Gunakan ini untuk Opsi B (Jalankan cell ini alih-alih uvicorn)
# !python kaggle/kaggle_oracle.py